# Load model weights to GPU

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B-Base"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model
model = AutoModelForCausalLM.from_pretrained(model_name)

# Example of generating text (optional)
inputs = tokenizer("Hello, world!", return_tensors="pt")
outputs = model.generate(inputs["input_ids"], max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Hello, world! is a song by the American rock band The Who. It was released in 1968 as the first single from their album The Who's Magical Mystery Tour. The song was written by John The Wellman, who also wrote the song "Hello, World!" for the 1968 film The Who's Magical Mystery Tour. The Who's Magical Mystery Tour was the first album to be released on the newly formed Columbia Records label, and it was the first album to be released on


In [2]:
print(inputs)
print(outputs)

{'input_ids': tensor([[9707,   11, 1879,    0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}
tensor([[ 9707,    11,  1879,     0,   374,   264,  5492,   553,   279,  3693,
          6946,  7055,   576, 10479,    13,  1084,   572,  5880,   304,   220,
            16,    24,    21,    23,   438,   279,  1156,  3175,   504,   862,
          8017,   576, 10479,   594, 72710, 48005, 14644,    13,   576,  5492,
           572,  5326,   553,  3757,   576,  8325,  1515,    11,   879,  1083,
          6139,   279,  5492,   330,  9707,    11,  4337,  8958,   369,   279,
           220,    16,    24,    21,    23,  4531,   576, 10479,   594, 72710,
         48005, 14644,    13,   576, 10479,   594, 72710, 48005, 14644,   572,
           279,  1156,  8017,   311,   387,  5880,   389,   279, 13631, 14122,
         18796, 21566,  2383,    11,   323,   432,   572,   279,  1156,  8017,
           311,   387,  5880,   389]])


In [3]:
del model

# Send request to LLM API

In [4]:
from openai import OpenAI

API_KEY = '<YOUR_API_KEY>'
# Initialize the client to connect to OpenAI API service
client = OpenAI(
    api_key=API_KEY
)

# Request a chat completion with streaming enabled
response = client.chat.completions.create(
    model='<MODEL_NAME>', # Specify the model name
    messages=[
        {'role': 'user', 'content': 'Hello world.'}
    ], stream = False,
)

# Process and print the streamed response
print(response.choices[0].message.content)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: <YOUR_AP**KEY>. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

# Host LLM models with Ollama

In [ ]:
# Model selection
MODEL_NAME = "llama3:8b" # TODO: For now, use "llama3:8b"
%env OLLAMA_CONTEXT_LENGTH=8000
%env OLLAMA_HOST=0.0.0.0
%env OLLAMA_KEEP_ALIVE=-1

In [ ]:
!apt-get install -y lshw pciutils
!nvcc --version
!nvidia-smi

# Print available memory
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"\nAvailable RAM: {ram_gb:.1f} GB")

In [ ]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import requests
import threading

# Start ollama serve in a background thread
def start_ollama():
    subprocess.call(['ollama', 'serve'])

ollama_thread = threading.Thread(target=start_ollama)
ollama_thread.daemon = True
ollama_thread.start()

# Pull model (this also verifies Ollama CLI is ready)
!ollama pull {MODEL_NAME}

# Wait for Ollama HTTP API to be ready
def wait_for_ollama(timeout=60):
    for i in range(timeout):
        try:
            r = requests.get("http://localhost:11434")
            if r.status_code in [200, 404]:
                print(f"Ollama is up (after {i+1}s).")
                return
        except requests.exceptions.ConnectionError:
            pass
        print(f"Waiting for Ollama to start... {i+1}s")
        time.sleep(1)
    raise RuntimeError("Ollama did not start in time.")

wait_for_ollama()

# Sending requests

### Streaming Enable

In [ ]:
from openai import OpenAI

# Initialize the client to connect to the local Ollama instance
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama', # The api_key is required but ignored by Ollama
)

# Request a chat completion with streaming enabled
response_stream = client.chat.completions.create(
    model=MODEL_NAME, # Specify the model name
    messages=[
        {'role': 'user', 'content': 'Explain the concept of quantum physics in a simple way, like I am a beginner.'}
    ],
    stream=True, # Enable streaming
)

# Process and print the streamed response
print("AI Response (Streaming):")
for chunk in response_stream:
    # Extract the token and print it
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

print()

### Streaming disable

In [ ]:
# Make a chat completion request with streaming disabled
response = client.chat.completions.create(
    model=MODEL_NAME, # Specify the model, e.g., llama3
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what the difference is between a Llama 3 8B and 70B model in a short sentence."}
    ], # TODO: Set stream to False to disable streaming
)

# Print the complete response content
print(response.choices[0].message.content)

# Expose as API

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import re

# Run cloudflared tunnel in background and get the public URL
cloudflared_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cloudflared_proc.stdout:
    print(line.strip())
    match = re.search(r'(https://.*\.trycloudflare\.com)', line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f"\nPublic URL for Ollama:\n{public_url}")
else:
    raise RuntimeError("Could not find public Cloudflare URL.")

In [ ]:
client_new = OpenAI(
    base_url=f"{public_url}/v1",
    api_key='ollama', # The api_key is required but ignored by Ollama
)

response = client_new.chat.completions.create(
    model=MODEL_NAME, # Specify the model, e.g., llama3
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what the difference is between a Llama 3 8B and 70B model in a short sentence."}
    ],
    stream=False, # Set stream to False to disable streaming
)

# Print the complete response content
print(response.choices[0].message.content)

# TODO: print out the number of input/output tokens

# Experiment with temperature and top_p

In [ ]:
def test_llama3(client, prompt, temp, top_p):
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a creative writer."},
            {"role": "user", "content": prompt}
        ],
        # TODO: pass the top_p and temperature params
    )
    return completion.choices[0].message.content

prompt = "Write a one-sentence story about a robot."

# 1. Deterministic/Conservative (Low Temp/Low Top_p)
print("--- Conservative ---")
print(test_llama3(client_new, prompt, 0.1, 0.1))

# 2. Creative/Random (High Temp/High Top_p)
print("\n--- Creative ---")
print(test_llama3(client_new, prompt, 1.5, 0.95))

# Chatbot loop

### Without context

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": user_input}
        ],
        max_tokens=300
    )

    print("Bot:", response.choices[0].message.content)

### With context

In [ ]:
messages = []

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    messages.append({"role": "user", "content": user_input})
    messages = messages[-6:]  # keep last 3 Q&A pairs (3 user + 3 assistant)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=300
    )

    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    print("Bot:", reply)